# Run Performance Benchmarks with EvalHub + GuideLLM

This notebook runs **performance benchmarks** through [EvalHub](https://github.com/eval-hub/eval-hub) using [GuideLLM](https://github.com/vllm-project/guidellm) as the benchmarking framework. Results are automatically tracked in **MLflow**.

**Architecture:**
```
Notebook (SDK) → EvalHub Service → GuideLLM Adapter Pod → MaaS / vLLM
                               └→ MLflow (automatic tracking)
```

**What we'll do:**
1. Configure endpoints and initialize EvalHub client
2. Submit a quick synchronous benchmark (baseline TTFT, ITL, tok/s)
3. Submit a sweep benchmark (latency-vs-load curve)
4. Monitor job progress and view results
5. Compare results via MLflow

**Pattern reference:** [rhoai-lmeval-builder-lab/4_eval_hub_benchmark](https://github.com/hyogrin/rhoai-lmeval-builder-lab)

**Prerequisites:**
- EvalHub service deployed in `demo` namespace (with GuideLLM provider registered)
- MaaS gateway with `/health` pass-through HTTPRoute configured
- Model deployed and accessible via MaaS
- EvalHub SA (`demo:evalhub-service`) must have permission to create ConfigMaps/Pods in the `demo` namespace

> **RBAC Note:** EvalHub creates ConfigMaps and Pods to run GuideLLM adapter jobs.
> If using a different namespace as tenant, grant permissions:
> ```bash
> oc create rolebinding evalhub-manager -n <namespace> \\
>   --clusterrole=admin --serviceaccount=demo:evalhub-service
> ```

## Step 1: Configuration

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'eval-hub-sdk'])

import os, time
from importlib.metadata import version
print(f"eval-hub-sdk: {version('eval-hub-sdk')}")

In [ ]:
cluster_domain = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
).stdout.strip()

NAMESPACE = "demo"
EVALHUB_URL = f"https://evalhub-{NAMESPACE}.{cluster_domain}"
MAAS_URL = f"https://maas-api.{cluster_domain}/v1"
MODEL_NAME = "qwen25-coder-7b"
PROCESSOR = "Qwen/Qwen2.5-Coder-7B-Instruct"
AUTH_TOKEN = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True).stdout.strip()

print(f"Namespace:  {NAMESPACE}")
print(f"EvalHub:    {EVALHUB_URL}")
print(f"MaaS:       {MAAS_URL}")
print(f"Model:      {MODEL_NAME}")
print(f"Processor:  {PROCESSOR}")

### Verify Prerequisites: RBAC & Connectivity

In [ ]:
import urllib.request, ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

checks = {
    "EvalHub health": f"https://evalhub-{NAMESPACE}.{cluster_domain}/health",
    "MaaS /health (GuideLLM validation)": f"https://maas-api.{cluster_domain}/health",
    "MaaS /v1/models": f"https://maas-api.{cluster_domain}/v1/models",
}

for name, url in checks.items():
    try:
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {AUTH_TOKEN}"})
        resp = urllib.request.urlopen(req, context=ctx, timeout=10)
        print(f"  [OK] {name} ({resp.status})")
    except Exception as e:
        print(f"  [FAIL] {name}: {e}")

# Check EvalHub SA RBAC
rbac_check = subprocess.run(
    ["oc", "auth", "can-i", "create", "configmaps", "-n", NAMESPACE,
     "--as=system:serviceaccount:demo:evalhub-service"],
    capture_output=True, text=True
)
can_create = rbac_check.stdout.strip() == "yes"
print(f"  [{'OK' if can_create else 'FAIL'}] EvalHub SA can create configmaps in {NAMESPACE}")

if not can_create:
    print(f"\n  Fix with: oc create rolebinding evalhub-manager -n {NAMESPACE} ")
    print(f"    --clusterrole=admin --serviceaccount=demo:evalhub-service")

## Step 2: Initialize EvalHub Client

In [ ]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(
    url=MAAS_URL,
    name=MODEL_NAME,
)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target: {model.name} @ {model.url}")

## Step 3: Verify GuideLLM Provider

In [ ]:
providers = client.providers.list()
guidellm_provider = None
for p in providers:
    if 'guidellm' in p.name.lower():
        guidellm_provider = p
        break

if guidellm_provider:
    print(f"GuideLLM provider found: {guidellm_provider.name} (id={guidellm_provider.resource.id})")
    print(f"Benchmarks:")
    for bm in guidellm_provider.benchmarks:
        print(f"  - {bm.id:30s} {bm.name}")
    GUIDELLM_PROVIDER_ID = guidellm_provider.resource.id
else:
    raise RuntimeError("GuideLLM provider not registered. Run 0_setup first.")

## Step 4: Helper — Monitor Job Progress

In [ ]:
TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
    JobStatus.PARTIALLY_FAILED,
}


def wait_for_job(client, job_id, poll_interval=10, max_wait=600):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.effective_state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.state.value}" for b in status.status.benchmarks]
            bm_info = f" | {', '.join(bm_states)}"

        print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    return status


def display_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        if job.status and job.status.message:
            print(f"  Message: {job.status.message.message}")
        return

    print("=" * 70)
    print("  BENCHMARK RESULTS")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:35s} = {value:.4f}")
                else:
                    print(f"    {name:35s} = {value}")
    print("\n" + "=" * 70)

---

## Benchmark 1: Quick Synchronous Test

Single-user baseline — measures **TTFT**, **ITL**, and per-request latency without concurrency.

In [ ]:
from datetime import datetime
RUN_TAG = datetime.now().strftime('%m%d-%H%M')

quick_request = JobSubmissionRequest(
    name=f"guidellm-quick-{MODEL_NAME}-{RUN_TAG}",
    description=f"Quick single-user performance test for {MODEL_NAME}",
    tags=["performance", "guidellm", "quick", "coding-assistant"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="quick_perf_test",
            provider_id=GUIDELLM_PROVIDER_ID,
            parameters={
                "profile": "synchronous",
                "max_seconds": 60,
                "data": "prompt_tokens=512,output_tokens=256",
                "request_type": "chat_completions",
                "processor": PROCESSOR,
                "backend_kwargs": {"verify": False},
            },
        ),
    ],
    experiment=ExperimentConfig(
        name=f"coding-assistant-perf/{MODEL_NAME}",
        tags=[
            ExperimentTag(key="model_family", value="qwen2.5-coder"),
            ExperimentTag(key="benchmark_tool", value="guidellm"),
            ExperimentTag(key="deployment", value="rhoai-maas"),
            ExperimentTag(key="evaluation_type", value="performance"),
        ],
    ),
)

quick_job = client.jobs.submit(quick_request)

print(f"Job submitted!")
print(f"  Job ID:  {quick_job.id}")
print(f"  Name:    {quick_job.name}")
print(f"  State:   {quick_job.state.value}")

In [ ]:
completed_quick = wait_for_job(client, quick_job.id)
display_results(completed_quick)

---

## Benchmark 2: Sweep — Latency vs Load Curve

Sweeps from low to high concurrency to discover the optimal throughput-latency tradeoff.

In [ ]:
sweep_request = JobSubmissionRequest(
    name=f"guidellm-sweep-{MODEL_NAME}-{RUN_TAG}",
    description=f"Sweep benchmark for {MODEL_NAME} — latency vs load curve",
    tags=["performance", "guidellm", "sweep", "coding-assistant"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="sweep",
            provider_id=GUIDELLM_PROVIDER_ID,
            parameters={
                "profile": "sweep",
                "max_seconds": 60,
                "data": "prompt_tokens=512,output_tokens=256",
                "request_type": "chat_completions",
                "processor": PROCESSOR,
                "backend_kwargs": {"verify": False},
            },
        ),
    ],
    experiment=ExperimentConfig(
        name=f"coding-assistant-perf/{MODEL_NAME}",
        tags=[
            ExperimentTag(key="model_family", value="qwen2.5-coder"),
            ExperimentTag(key="benchmark_tool", value="guidellm"),
            ExperimentTag(key="evaluation_type", value="sweep"),
        ],
    ),
)

sweep_job = client.jobs.submit(sweep_request)
print(f"Sweep job submitted: {sweep_job.id}")
print(f"This will take ~5-7 minutes...")

completed_sweep = wait_for_job(client, sweep_job.id, poll_interval=15, max_wait=900)
display_results(completed_sweep)

---

## Benchmark 3: Throughput — Maximum Capacity

Finds the server's maximum requests-per-second by increasing load until saturation.

In [ ]:
throughput_request = JobSubmissionRequest(
    name=f"guidellm-throughput-{MODEL_NAME}-{RUN_TAG}",
    description=f"Max throughput discovery for {MODEL_NAME}",
    tags=["performance", "guidellm", "throughput", "coding-assistant"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="throughput",
            provider_id=GUIDELLM_PROVIDER_ID,
            parameters={
                "profile": "throughput",
                "max_seconds": 120,
                "data": "prompt_tokens=512,output_tokens=256",
                "request_type": "chat_completions",
                "processor": PROCESSOR,
                "backend_kwargs": {"verify": False},
                "detect_saturation": True,
            },
        ),
    ],
    experiment=ExperimentConfig(
        name=f"coding-assistant-perf/{MODEL_NAME}",
        tags=[
            ExperimentTag(key="model_family", value="qwen2.5-coder"),
            ExperimentTag(key="benchmark_tool", value="guidellm"),
            ExperimentTag(key="evaluation_type", value="throughput"),
        ],
    ),
)

# Uncomment to run:
# throughput_job = client.jobs.submit(throughput_request)
# print(f"Throughput job submitted: {throughput_job.id}")
# completed_throughput = wait_for_job(client, throughput_job.id, poll_interval=15, max_wait=900)
# display_results(completed_throughput)

---

## Step 5: Compare Results

In [ ]:
jobs_list = client.jobs.list()

print(f"Total Jobs: {jobs_list.total_count}")
print("=" * 90)
print(f"{'State':>16s}  {'Job ID':12s}  {'Name':35s}  Benchmarks")
print("-" * 90)
for j in jobs_list.items:
    state = j.effective_state.value
    bms = [b.id for b in j.benchmarks] if j.benchmarks else []
    print(f"{state:>16s}  {j.id[:12]:12s}  {j.name[:35]:35s}  {bms}")

In [ ]:
def collect_perf_comparison(client):
    """Collect GuideLLM metrics from completed jobs."""
    jobs_list = client.jobs.list()
    completed = [j for j in jobs_list.items if j.effective_state == JobStatus.COMPLETED]

    rows = []
    for j in completed:
        if not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.metrics:
                row = {"job": j.name, "benchmark": bm.id}
                row.update(bm.metrics)
                rows.append(row)
    return rows


rows = collect_perf_comparison(client)

if rows:
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        key_metrics = [c for c in df.columns if c in [
            'requests_per_second', 'output_tokens_per_second',
            'prompt_tokens_per_second', 'mean_ttft_ms', 'mean_itl_ms'
        ]]
        display(df[['job', 'benchmark'] + key_metrics].style.format(
            {m: '{:.2f}' for m in key_metrics}
        ))
    except ImportError:
        for row in rows:
            print(f"  {row['job']:35s} | {row['benchmark']:20s} | rps={row.get('requests_per_second','?'):.2f}")
else:
    print("No completed GuideLLM jobs yet.")

---

## Step 6: MLflow Integration

Access evaluation results directly from MLflow for advanced analysis, comparison, and visualization.
EvalHub automatically tracks all benchmark results to MLflow.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow'])

import mlflow
from mlflow.tracking.request_header.registry import _request_header_provider_registry

# RHOAI MLflow requires X-MLflow-Workspace header for multi-tenant access
class WorkspaceHeaderProvider:
    def in_context(self):
        return True
    def request_headers(self):
        return {'X-MLflow-Workspace': NAMESPACE}

_request_header_provider_registry.register(WorkspaceHeaderProvider)

MLFLOW_TRACKING_URI = f"https://mlflow-external-redhat-ods-applications.{cluster_domain}"
os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'
os.environ['MLFLOW_TRACKING_TOKEN'] = AUTH_TOKEN
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Workspace:           {NAMESPACE}")
print(f"mlflow version:      {mlflow.__version__}")

In [ ]:
# List MLflow experiments
experiments = mlflow.search_experiments()

print(f"MLflow Experiments ({len(experiments)}):")
print("=" * 70)
for exp in experiments:
    marker = "  <<<" if "perf" in exp.name.lower() or "guidellm" in str(exp.tags).lower() else ""
    print(f"  [{exp.experiment_id:>3s}] {exp.name}{marker}")

In [ ]:
# EvalHub adapter automatically logs to MLflow (run + trace).
# Verify the run was created:
EXPERIMENT_NAME = f"coding-assistant-perf/{MODEL_NAME}"

if completed_quick and completed_quick.results:
    for bm in completed_quick.results.benchmarks:
        if bm.mlflow_run_id:
            print(f"MLflow Run ID (auto-created by adapter): {bm.mlflow_run_id}")
        else:
            print("WARNING: No mlflow_run_id — adapter may not have MLflow tracing enabled.")
    if completed_quick.results.mlflow_experiment_url:
        print(f"MLflow Experiment URL: {completed_quick.results.mlflow_experiment_url}")
else:
    print("Submit and complete a benchmark first (Step 4).")

### Query MLflow Runs

In [ ]:
from IPython.display import display

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if exp:
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=["start_time DESC"],
        max_results=20,
    )

    if not runs.empty:
        print(f"MLflow Runs in '{EXPERIMENT_NAME}': {len(runs)}")
        metric_cols = [c for c in runs.columns if c.startswith('metrics.')]
        display_cols = ['run_id', 'status', 'start_time'] + metric_cols[:6]
        available = [c for c in display_cols if c in runs.columns]
        display(runs[available].head(10))
    else:
        print(f"No runs found in experiment '{EXPERIMENT_NAME}'.")
else:
    print(f"Experiment '{EXPERIMENT_NAME}' not found.")
    print("Available experiments:")
    for e in mlflow.search_experiments():
        print(f"  - {e.name}")

### Visualize Metrics

In [ ]:
try:
    import matplotlib.pyplot as plt

    exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if exp:
        runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
        metric_cols = [c for c in runs.columns if c.startswith('metrics.')]

        if not runs.empty and metric_cols:
            metrics_df = runs[metric_cols].dropna(axis=1, how='all')
            metrics_df.columns = [c.replace('metrics.', '') for c in metrics_df.columns]

            fig, ax = plt.subplots(figsize=(12, 6))
            metrics_df.plot(kind='bar', ax=ax)
            ax.set_title(f'GuideLLM Benchmark Metrics — {EXPERIMENT_NAME}')
            ax.set_ylabel('Value')
            ax.set_xlabel('Run')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.show()
        else:
            print('No metric data to visualize.')
    else:
        print(f"Experiment '{EXPERIMENT_NAME}' not found.")

except ImportError:
    print('matplotlib not installed. Install with: pip install matplotlib')
except Exception as e:
    print(f'Visualization failed: {e}')

---

## Summary

| Component | Role |
|-----------|------|
| **EvalHub** | Orchestration — schedules GuideLLM jobs as K8s Pods, manages lifecycle |
| **GuideLLM** | Benchmarking — synthetic workloads, precise latency/throughput measurements |
| **MLflow** | Tracking — params, metrics, artifacts for reproducibility and comparison |
| **MaaS /health** | HTTPRoute pass-through enabling GuideLLM backend validation |

**Available GuideLLM benchmarks via EvalHub:**

| Benchmark ID | Description |
|-------------|-------------|
| `quick_perf_test` | Fast synchronous test for baseline metrics |
| `sweep` | Auto-discover optimal load (latency-vs-throughput curve) |
| `throughput` | Find maximum server capacity |
| `concurrent` | Fixed concurrency stress test |
| `constant` | Steady-state load test |
| `poisson` | Realistic traffic simulation |
| `comprehensive_perf_test` | Full characterization across all profiles |

→ Continue to `3_capacity_planning.ipynb` for multi-replica projections and cost analysis.